### 연습 문제
- data 폴더 안에 'insurance.csv' 파일이 존재 
- 'charges' 종속 변수 
- torch의 다중 퍼셉트론을 이용하여 회귀형 모델을 생성 (epoch의 횟수는 300회 제한)
    - 종속변수도 2차원 데이터셋으로 변경 [ 1차 행렬 -> 2차 행렬 ]
    - 종속변수 데이터의 타입을 float32
- ML 모델로는 XGBoost을 이용하여 회귀 모델을 생성 
- R2 Score를 구해서 어떤 모델이 더 좋은 성능을 가지는가.
- XGBoost params 
    - n_estimator : [100, 200, 300]
    - max_depth = [3, 4, 5]
    - learning_rate : [0.01, 0.05, 0.1]
    - subsample : [0.8, 0.9, 1.0]

In [ ]:
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from xgboost import XGBRegressor
from sklearn.metrics import r2_score

import torch 
import torch.nn as nn
import torch.optim as optim


In [ ]:
df = pd.read_csv("../data/insurance.csv")

In [ ]:
df.info()

- age : 나이 
- sex : 성별
- bmi : 체질량수치
- children : 자녀의 수
- smoker : 흡연 여부
- region : 거주 지역
- charges : 의료비(target)

In [ ]:
df.head()

In [ ]:
df.describe()

In [ ]:
# 데이터 확인 
df['sex'].unique()

In [ ]:
df['smoker'].unique()

In [ ]:
df['region'].unique()

In [ ]:
# 범주형 데이터들을 더미화 
df = pd.get_dummies(df, columns = ['sex', 'smoker', 'region'], drop_first = True)
df.info()

In [ ]:
# 독립, 종속 데이터 분리 
x = df.drop('charges', axis = 1)
y = df['charges']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    x, y, test_size = 0.2, random_state = 42
)

In [ ]:
X_train.describe()

In [ ]:
# 딥러닝 다중 퍼셉트론 

scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)

# Tensor로 변환 
X_train_tensor = torch.tensor(X_train_sc, dtype = torch.float32)
X_test_tensor = torch.tensor(X_test_sc, dtype=torch.float32)

In [ ]:
# 1차행렬의 데이터를 2차 행렬로 변환 
y_train_tensor =  torch.tensor(y_train.values.reshape(-1, 1), dtype = torch.float32)
y_test_tensor = torch.tensor(y_test.values.reshape(-1, 1), dtype=torch.float32)

print(y_train_tensor.shape)

In [ ]:
# Tensor에서 에서 제공하는 데이터의 구조를 바꾸는 방식 
y_train_tensor2 = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(-1)
y_test_tensor2 = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(-1)

In [ ]:
# view()함수는 numpy의 reshape()과 유사한 기능을 하는 함수입니다.
torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)

In [ ]:
class Reg_Model(nn.Module):
    def __init__(self, _dim):
        super(Reg_Model, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(_dim, 64), 
            nn.ReLU(), 
            nn.Dropout(0.2), 
            nn.Linear(64, 32), 
            nn.ReLU(), 
            nn.Dropout(0.2),
            nn.Linear(32, 1)
        )
    def forward(self, x):
        return self.model(x)

In [ ]:
model = Reg_Model(X_train_tensor.shape[1])

In [ ]:
criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr = 0.01)

In [ ]:
epochs = 300

for epoch in range(epochs):
    # 모델의 예측값
    pred = model(X_train_tensor)
    # 예측값과 실제값의 차이를 계산
    loss = criterion(pred, y_train_tensor)
    # 기울기를 초기화 
    optimizer.zero_grad()
    # 역전파 계산 ( 자동 미분을 통해서 기울기의 방향을 알려준다. )
    loss.backward()
    # 가중치 업데이트(가중치를 역전파의 계산 방향으로 이동)
    optimizer.step()

    if (epoch + 1) % 30 == 0:
        print(f"Epoch {epoch + 1}/{epochs}, Loss: {round(loss.item(), 6)}")

In [ ]:
# 모델 평가 
model.eval()
# no_grad() 사용하는 이유는 가중치 업데이트를 잠시 중단 -> 계산이 필요가 없기 때문(메모리 사용 최적화)
with torch.no_grad():
    pred = model(X_test_tensor)
    r2 = r2_score(y_test_tensor, pred)
    print(f"R2 Score: {round(r2, 4)}")
    print("예측값 : ",pred[0], "실제값 : ", y_test_tensor[0])

In [ ]:
# ML (XGBoost)를 이용하여 모델 학습 및 예측

# Sclaer, Model Pipeline 구축
pipe = Pipeline([
    ('std', StandardScaler()),
    ('xgb', XGBRegressor(random_state = 42))
])

params = {
    'xgb__n_estimators' : [100, 200, 300], 
    'xgb__learning_rate' : [0.01, 0.05, 0.1],
    'xgb__max_depth' : [3, 4, 5], 
    'xgb__subsample' : [0.8, 0.9, 1.0]
}

cv = KFold(n_splits = 5, shuffle = True, random_state = 42)

grid = GridSearchCV(
    pipe, 
    param_grid = params, 
    cv = cv, 
    n_jobs = -1, 
    scoring = 'r2'
)


In [ ]:
grid.fit(X_train.values, y_train.values)

In [ ]:
# 최적의 파라미터 확인 
print(grid.best_params_)

In [16]:
pred = grid.predict(X_test)

r2 = r2_score(y_test, pred)
print(f"R2 Score: {round(r2, 4)}")
print("예측값 : ",pred[0], "실제값 : ", y_test.values[0])

R2 Score: 0.8832
예측값 :  10831.94 실제값 :  9095.06825


c:\Users\ekfla\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\utils\validation.py:2684: UserWarning: X has feature names, but StandardScaler was fitted without feature names
  warnings.warn(
